# Exploratory Data Analysis, Step by Step

Notebook for **Chapter 7** of the *analyst* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/analyst/ch07.html

Tip: change a number, predict the output, then run the cell again.

## 7.1 Context, Grain and the First Look

In [ ]:
import numpy as np, pandas as pd

def make_orders(seed=7, n_customers=1500):
    rng = np.random.default_rng(seed)
    cust = pd.DataFrame({
        "customer_id": [f"C{i:04d}" for i in range(n_customers)],
        "signup_date": pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 180, n_customers), unit="D"),
        "channel": rng.choice(["organic", "ads", "referral", "email"], n_customers, p=[.4, .3, .2, .1]),
        "city": rng.choice(["Pune", "Mumbai", "Delhi", "Bengaluru", "Chennai"], n_customers, p=[.15, .25, .25, .25, .10]),
    })
    rows = []
    oid = 1000
    for r in cust.itertuples():
        k = rng.poisson(1.6 if r.channel != "ads" else 1.1)
        for _ in range(k):
            od = r.signup_date + pd.Timedelta(days=int(rng.exponential(30)))
            if od > pd.Timestamp("2025-12-31"): continue
            cat = rng.choice(["electronics", "fashion", "grocery", "home"], p=[.2, .35, .3, .15])
            base = {"electronics": 6.8, "fashion": 6.0, "grocery": 5.0, "home": 6.3}[cat]
            amount = round(float(rng.lognormal(base, 0.5)), 2)
            qty = int(rng.integers(1, 5))
            status = rng.choice(["delivered", "returned", "cancelled"], p=[.86, .08, .06])
            rows.append((oid, r.customer_id, od, cat, qty, amount, status)); oid += 1
    o = pd.DataFrame(rows, columns=["order_id", "customer_id", "order_date", "category", "qty", "amount", "status"])
    idx = rng.choice(o.index, 60, replace=False); o.loc[idx, "amount"] = np.nan
    idx = rng.choice(o.index, 15, replace=False); o.loc[idx, "qty"] = -1
    idx = rng.choice(o.index, 8, replace=False); o.loc[idx, "amount"] *= 100
    o = pd.concat([o, o.sample(12, random_state=1)], ignore_index=True)
    o["category"] = o["category"].where(rng.random(len(o)) > .05, o["category"].str.upper())
    return cust, o.sample(frac=1, random_state=3).reset_index(drop=True)

In [ ]:
import pandas as pd, numpy as np
from gen import make_orders
cust, o = make_orders()
print(o.shape, cust.shape)
print(o.dtypes)

## 7.2 Data Quality Scan and Cleaning

In [ ]:
print(o.isna().sum())
print("exact duplicate rows:", o.duplicated().sum())
print("negative or zero qty:", (o.qty <= 0).sum())
print(o.category.value_counts())

order_id        0
customer_id     0
order_date      0
category        0
qty             0
amount         61
status          0
dtype: int64
exact duplicate rows: 12
negative or zero qty: 15
category
fashion        735
grocery        661
electronics    437
home           310
GROCERY         31
FASHION         31
ELECTRONICS     21
HOME            18
Name: count, dtype: int64


In [ ]:
o = o.drop_duplicates()
o["category"] = o["category"].str.lower()
o.loc[o.qty <= 0, "qty"] = np.nan
cat_median = o.groupby("category")["amount"].transform("median")
o["ratio"] = o["amount"] / cat_median
print(o.loc[o.ratio >= 15, ["order_id", "category", "amount", "ratio"]].sort_values("ratio").round(1))
o.loc[o.ratio >= 15, "amount"] /= 100
o = o.drop(columns="ratio")
print(len(o), o.amount.isna().sum(), o.qty.isna().sum())

      order_id     category    amount  ratio
2226      2633      grocery    3566.0   23.4
944       2072      fashion   21933.0   53.0
752       1675         home   47982.0   92.3
345       3043      grocery   15105.0   99.0
595       2282  electronics  111473.0  131.6
892       1796         home   89659.0  172.5
659       1870      grocery   30409.0  199.4
2232 60 15


## 7.3 Missing Data: Pattern, Mechanism, Treatment

In [ ]:
print(o.assign(miss=o.amount.isna()).groupby("category")["miss"].mean().round(3))

category
electronics    0.029
fashion        0.022
grocery        0.030
home           0.028
Name: miss, dtype: float64


## 7.4 Univariate Analysis: One Column at a Time

In [ ]:
a = o["amount"].dropna()
print(a.describe().round(1))
print("skew", round(a.skew(), 2), "log-skew", round(np.log(a).skew(), 2), "kurtosis", round(a.kurt(), 2))
print(a.quantile([.5, .9, .99]).round(1))
print(pd.cut(a, bins=[0, 100, 250, 500, 1000, 5000]).value_counts().sort_index())

count    2172.0
mean      495.6
std       410.3
min        33.3
25%       201.7
50%       383.8
75%       654.7
max      3866.0
Name: amount, dtype: float64
skew 2.09 log-skew -0.16 kurtosis 7.42
0.50     383.8
0.90    1013.7
0.99    1927.5
Name: amount, dtype: float64
amount
(0, 100]        130
(100, 250]      588
(250, 500]      631
(500, 1000]     598
(1000, 5000]    225
Name: count, dtype: int64


## 7.5 Bivariate and Multivariate Analysis

In [ ]:
print(o.groupby("category")["amount"].agg(["count", "median", "mean"]).round(1))
print(pd.crosstab(o.category, o.status, normalize="index").round(3))
print(o[["qty", "amount"]].corr().round(3))
j = o.merge(cust, on="customer_id", how="left", validate="many_to_one")
print(j.groupby("channel").agg(orders=("order_id", "count"), aov=("amount", "mean")).round(1))

             count  median   mean
category                         
electronics    443   847.2  967.9
fashion        743   413.6  461.8
grocery        668   152.4  169.9
home           318   517.7  600.8
status       cancelled  delivered  returned
category                                   
electronics      0.081      0.851     0.068
fashion          0.054      0.846     0.100
grocery          0.054      0.871     0.075
home             0.064      0.893     0.043
          qty  amount
qty     1.000   0.011
amount  0.011   1.000
          orders    aov
channel                
ads          475  489.7
email        249  502.7
organic     1013  499.5
referral     495  490.0


In [ ]:
from scipy import stats
ct = pd.crosstab(o.category, o.status == "returned")
chi2, p, dof, exp = stats.chi2_contingency(ct)
print(round(chi2, 2), round(p, 4), dof)

11.5 0.0093 3


In [ ]:
m = o.groupby(o.order_date.dt.to_period("M")).agg(orders=("order_id", "count"), revenue=("amount", "sum")).round(0)
per = o.groupby("customer_id")["order_id"].count()
print(m)
print(per.value_counts().sort_index())
print("customers with >=1 order:", len(per), "of", len(cust))
print("repeat rate:", round((per >= 2).mean(), 3))

            orders  revenue
order_date                 
2025-01        144  76045.0
2025-02        233  115867.0
2025-03        348  162274.0
2025-04        364  164132.0
2025-05        390  194888.0
2025-06        393  186105.0
2025-07        222  114926.0
2025-08         95   39809.0
2025-09         31   16305.0
2025-10          9    3485.0
2025-11          2     648.0
2025-12          1    2022.0
order_id
1    521
2    358
3    178
4     77
5     19
6      5
7      4
Name: count, dtype: int64
customers with >=1 order: 1162 of 1500
repeat rate: 0.552


## 7.6 From Hypotheses to Documented Insights (and Profiling Tools)

In [ ]:
def quick_scan(df, key):
    return {
        "rows": len(df),
        "dup_keys": int(df[key].duplicated().sum()),
        "missing_pct": (df.isna().mean() * 100).round(1).to_dict(),
    }

## 7.7 Regression for Analysts: Lines, Coefficients and Honest Reading

In [ ]:
import numpy as np
from scipy import stats
ad = np.array([10, 20, 30, 40, 50, 60.])
sales = np.array([120, 150, 160, 190, 210, 215.])
xm, ym = ad.mean(), sales.mean()
sxy = ((ad - xm) * (sales - ym)).sum()
sxx = ((ad - xm) ** 2).sum()
b1 = sxy / sxx
b0 = ym - b1 * xm
print("means:", xm, round(ym, 2), " Sxy:", sxy, " Sxx:", sxx)
print("slope:", round(b1, 4), " intercept:", round(b0, 3))
res = sales - (b0 + b1 * ad)
sse = (res ** 2).sum(); sst = ((sales - ym) ** 2).sum()
print("residuals:", res.round(2))
print("SSE:", round(sse, 2), " SST:", round(sst, 2), " R-squared:", round(1 - sse / sst, 4))
r = stats.linregress(ad, sales)                     # the library check
print("linregress:", round(r.slope, 4), round(r.intercept, 3), round(r.rvalue ** 2, 4))

means: 35.0 174.17  Sxy: 3425.0  Sxx: 1750.0
slope: 1.9571  intercept: 105.667
residuals: [-5.24  5.19 -4.38  6.05  6.48 -8.1 ]
SSE: 217.62  SST: 6920.83  R-squared: 0.9686
linregress: 1.9571 105.667 0.9686


In [ ]:
import numpy as np
from scipy import stats
ad = np.array([10, 20, 30, 40, 50, 60.]); sales = np.array([120, 150, 160, 190, 210, 215.])
n = 6; xm = ad.mean(); sxx = ((ad - xm) ** 2).sum()
b1 = ((ad - xm) * (sales - sales.mean())).sum() / sxx; b0 = sales.mean() - b1 * xm
s2 = ((sales - b0 - b1 * ad) ** 2).sum() / (n - 2)
t = stats.t.ppf(0.975, n - 2)
se_b1 = np.sqrt(s2 / sxx)
print("s2:", round(s2, 2), " se(slope):", round(se_b1, 4), " t:", round(b1 / se_b1, 2), " t crit:", round(t, 4))
print("95% CI for slope:", round(b1 - t * se_b1, 3), "to", round(b1 + t * se_b1, 3))
for x0 in (35, 100):
    fit = b0 + b1 * x0
    ci = t * np.sqrt(s2 * (1 / n + (x0 - xm) ** 2 / sxx))          # for the AVERAGE week at this spend
    pi = t * np.sqrt(s2 * (1 + 1 / n + (x0 - xm) ** 2 / sxx))      # for ONE new week
    print("spend", x0, "forecast", round(fit, 1), " mean +/-", round(ci, 1), " single week +/-", round(pi, 1))

s2: 54.4  se(slope): 0.1763  t: 11.1  t crit: 2.7764
95% CI for slope: 1.468 to 2.447
spend 35 forecast 174.2  mean +/- 8.4  single week +/- 22.1
spend 100 forecast 301.4  mean +/- 32.9  single week +/- 38.8


In [ ]:
import numpy as np, pandas as pd
from scipy import stats
rng = np.random.default_rng(12)
n = 400
price = rng.uniform(8, 20, n).round(1)
ad = rng.uniform(0, 30, n).round(1)
discount = (rng.uniform(0, 1, n) < (0.15 + 0.02 * ad)).astype(int)
units = 60 - 2.0 * price + 0.5 * ad + 6 * discount + rng.normal(0, 5, n)
df = pd.DataFrame(dict(units=units.round(1), price=price, ad=ad, discount=discount))

def ols(X, y, names):
    X = np.column_stack([np.ones(len(y))] + X); k = X.shape[1]
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    res = y - X @ beta
    s2 = (res ** 2).sum() / (len(y) - k)
    se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    t = beta / se
    p = 2 * stats.t.sf(abs(t), len(y) - k)
    r2 = 1 - (res ** 2).sum() / ((y - y.mean()) ** 2).sum()
    adj = 1 - (1 - r2) * (len(y) - 1) / (len(y) - k)
    return pd.DataFrame(dict(coef=beta, se=se, t=t, p=p), index=["intercept"] + names).round(3), round(r2, 3), round(adj, 3)
y = df.units.values
full, r2, adj = ols([df.price.values, df.ad.values, df.discount.values], y, ["price", "ad", "discount"])
print(full); print("R2:", r2, " adjusted R2:", adj)

             coef     se       t    p
intercept  59.654  1.074  55.562  0.0
price      -2.029  0.069 -29.482  0.0
ad          0.537  0.030  17.735  0.0
discount    6.470  0.544  11.904  0.0
R2: 0.795  adjusted R2: 0.793


In [ ]:
import numpy as np, pandas as pd
from scipy import stats
rng = np.random.default_rng(12)
n = 400
price = rng.uniform(8, 20, n).round(1)
ad = rng.uniform(0, 30, n).round(1)
discount = (rng.uniform(0, 1, n) < (0.15 + 0.02 * ad)).astype(int)
units = 60 - 2.0 * price + 0.5 * ad + 6 * discount + rng.normal(0, 5, n)
df = pd.DataFrame(dict(units=units.round(1), price=price, ad=ad, discount=discount))

def ols(X, y, names):
    X = np.column_stack([np.ones(len(y))] + X); k = X.shape[1]
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    res = y - X @ beta
    s2 = (res ** 2).sum() / (len(y) - k)
    se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    t = beta / se
    p = 2 * stats.t.sf(abs(t), len(y) - k)
    r2 = 1 - (res ** 2).sum() / ((y - y.mean()) ** 2).sum()
    adj = 1 - (1 - r2) * (len(y) - 1) / (len(y) - k)
    return pd.DataFrame(dict(coef=beta, se=se, t=t, p=p), index=["intercept"] + names).round(3), round(r2, 3), round(adj, 3)
y = df.units.values
short, r2, adj = ols([df.price.values, df.discount.values], y, ["price", "discount"])
print(short); print("R2:", r2)
print(df.corr().round(2)["discount"])

             coef     se       t    p
intercept  65.718  1.362  48.268  0.0
price      -2.006  0.092 -21.795  0.0
discount   10.174  0.671  15.156  0.0
R2: 0.631
units       0.44
price       0.04
ad          0.39
discount    1.00
Name: discount, dtype: float64


In [ ]:
import numpy as np, pandas as pd
from scipy import stats
rng = np.random.default_rng(12)
n = 400
price = rng.uniform(8, 20, n).round(1)
ad = rng.uniform(0, 30, n).round(1)
discount = (rng.uniform(0, 1, n) < (0.15 + 0.02 * ad)).astype(int)
units = 60 - 2.0 * price + 0.5 * ad + 6 * discount + rng.normal(0, 5, n)
df = pd.DataFrame(dict(units=units.round(1), price=price, ad=ad, discount=discount))

def ols(X, y, names):
    X = np.column_stack([np.ones(len(y))] + X); k = X.shape[1]
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    res = y - X @ beta
    s2 = (res ** 2).sum() / (len(y) - k)
    se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    t = beta / se
    p = 2 * stats.t.sf(abs(t), len(y) - k)
    r2 = 1 - (res ** 2).sum() / ((y - y.mean()) ** 2).sum()
    adj = 1 - (1 - r2) * (len(y) - 1) / (len(y) - k)
    return pd.DataFrame(dict(coef=beta, se=se, t=t, p=p), index=["intercept"] + names).round(3), round(r2, 3), round(adj, 3)
def vif(cols):
    out = {}
    for j, c in enumerate(cols):
        others = [o for i, o in enumerate(cols) if i != j]
        _, r2, _ = ols([df[o].values for o in others], df[c].values, others)
        out[c] = float(round(1 / (1 - r2), 2))
    return out
df["ad_copy"] = df.ad + rng.normal(0, 0.5, n)
print(vif(["price", "ad", "discount"]))
bad, r2, adj = ols([df.price.values, df.ad.values, df.ad_copy.values], df.units.values, ["price", "ad", "ad_copy"])
print(bad); print("R2:", r2)
print(vif(["price", "ad", "ad_copy"]))

{'price': 1.0, 'ad': 1.18, 'discount': 1.18}
             coef     se       t      p
intercept  59.956  1.245  48.158  0.000
price      -1.995  0.080 -24.996  0.000
ad          1.877  0.553   3.396  0.001
ad_copy    -1.199  0.551  -2.177  0.030
R2: 0.724
{'price': 1.0, 'ad': 333.33, 'ad_copy': 333.33}


In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(3)
n = 5000
mobile = (rng.uniform(size=n) < 0.6).astype(int)
coupon = (rng.uniform(size=n) < 0.5).astype(int)
visits = rng.poisson(3, n)
p = 1 / (1 + np.exp(-(-3.0 - 0.4 * mobile + 0.7 * coupon + 0.25 * visits)))
y = (rng.uniform(size=n) < p).astype(int)
print(pd.DataFrame(dict(coupon=coupon, conv=y)).groupby("coupon")["conv"].agg(["mean", "size"]).round(4))

X = np.column_stack([np.ones(n), mobile, coupon, visits])
beta = np.zeros(4)
for it in range(25):                                   # Newton steps, usually 5 to 8
    mu = 1 / (1 + np.exp(-X @ beta)); W = mu * (1 - mu)
    step = np.linalg.solve(X.T @ (X * W[:, None]), X.T @ (y - mu))
    beta += step
    if np.abs(step).max() < 1e-10: break
se = np.sqrt(np.diag(np.linalg.inv(X.T @ (X * W[:, None]))))
table = pd.DataFrame(dict(coef=beta, se=se, odds_ratio=np.exp(beta), lo=np.exp(beta - 1.96 * se), hi=np.exp(beta + 1.96 * se)),
                     index=["intercept", "mobile", "coupon", "visits"]).round(3)
print("iterations:", it + 1); print(table)
sk = LogisticRegression(C=1e9, max_iter=1000).fit(X[:, 1:], y)
print("sklearn:", sk.intercept_.round(3), sk.coef_.round(3))
pr = lambda m, c, v: 1 / (1 + np.exp(-(beta @ np.array([1, m, c, v]))))
print("desktop, 3 visits: coupon", round(pr(0, 1, 3), 4), " no coupon", round(pr(0, 0, 3), 4))

          mean  size
coupon              
0       0.0818  2468
1       0.1647  2532
iterations: 7
           coef     se  odds_ratio     lo     hi
intercept -2.93  0.123       0.053  0.042  0.068
mobile    -0.45  0.088       0.638  0.537  0.758
coupon     0.80  0.092       2.225  1.857  2.665
visits     0.23  0.024       1.259  1.201  1.319
sklearn: [-2.93] [[-0.45   0.798  0.23 ]]
desktop, 3 visits: coupon 0.1916  no coupon 0.0963


## Chapter recap

## Practice set

Problem 1 A column has values 10, 12, 11, 13, 200. Mean and median? Which is typical?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/analyst/ch07.html